In [6]:
import math
import numpy as np
import geopandas as gpd
import rasterio as rio
from shapely.geometry import box
from tqdm import tqdm
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
# Country rasters define the grid (extent, transform, shape) for each
# resolution. We only need them as spatial templates for the output rasters.
# RESOLUTIONS = ['1km', '5km', '10km', '50km']
RESOLUTIONS = ['1km']
country_rasters = {
    res: rio.open(RESULTS / 'country-rasters' /  f'country-raster-eck4-{res}.tif')
    for res in RESOLUTIONS
}

In [4]:
CELL_SIZE_M = {'1km': 1e3, '2km': 2e3, '5km': 5e3, '10km': 1e4,
               '30km': 3e4, '50km': 5e4, '100km': 1e5}

def generate_box(i, j, res):
    """Return the Eckert IV polygon of cell (row=i, col=j) at given resolution."""
    cell_size = CELL_SIZE_M[res]
    xmin, ymin, xmax, ymax = country_rasters[res].bounds
    x_1 = xmin + j * cell_size
    x_2 = x_1 + cell_size
    y_2 = ymax - i * cell_size
    y_1 = y_2 - cell_size
    return box(x_1, y_1, x_2, y_2)

In [9]:
# Per-cell area cap (km²). Guards against any double-counting from overlapping
# polygons and ensures no cell exceeds its physical area.
CELL_AREA_CAP_KM2 = {'1km': 1, '2km': 4, '5km': 25, '10km': 100,
                     '30km': 900, '50km': 2500, '100km': 10000}

def make_quarry_area_grid(gdf, res):
    """Rasterize quarry polygons into a per-cell area grid (km²) at given resolution.

    For each polygon, identify the grid cells it overlaps via its bounding box,
    intersect with each candidate cell, and accumulate the intersection area
    (km²) into the output grid.
    """
    template = country_rasters[res]
    transform = template.transform
    grid = np.zeros(template.shape, dtype=float)
    fail_index = []
    total_area = 0

    for idx in tqdm(gdf.index):
        the_quarry = gdf.loc[[idx]]
        minx, miny, maxx, maxy = the_quarry.total_bounds

        # Map polygon bounds to row/col indices of overlapping cells.
        row_start, col_start = ~transform * (minx, maxy)
        row_end,   col_end   = ~transform * (maxx, miny)
        i_1, j_1 = math.floor(col_start), math.floor(row_start)
        i_2, j_2 = math.floor(col_end) + 1, math.floor(row_end) + 1

        try:
            for i in range(i_1, i_2 + 1):
                for j in range(j_1, j_2 + 1):
                    cell_box = generate_box(i, j, res)
                    a_km2 = round(the_quarry.intersection(cell_box).area.sum() / 1e6, 3)
                    grid[i, j] += a_km2
                    total_area += a_km2
        except Exception:
            fail_index.append(idx)

    # Cap each cell at its physical maximum area (sanity guard).
    cap = CELL_AREA_CAP_KM2[res]
    grid_capped = np.where(grid > cap, cap, grid)

    print(f"  raw total:    {total_area:,.2f} km²")
    print(f"  capped total: {np.nansum(grid_capped):,.2f} km²")
    print(f"  failed: {len(fail_index)}")
    return grid_capped

In [7]:
aggregate_quarries = gpd.read_file(DATA / 'global-aggregate-quarries-210101.gpkg')
print(aggregate_quarries.crs)        # should show ESRI:54012
print(len(aggregate_quarries))       # 136,666

PROJCS["World_Eckert_IV",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Eckert_IV"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","54012"]]
136666


In [8]:
# Build one output grid per resolution.
quarry_grids = {}
for res in RESOLUTIONS:
    print(f"Processing {res}...")
    quarry_grids[res] = make_quarry_area_grid(aggregate_quarries, res)

Processing 1km...


100%|██████████| 136666/136666 [01:56<00:00, 1176.11it/s]


  raw total:    19,139.11 km²
  capped total: 19,005.48 km²
  failed: 1


In [10]:
OUT_DIR = RESULTS / 'quarry-area-rasters'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for res, grid in quarry_grids.items():
    template = country_rasters[res]
    out_path = OUT_DIR / f'global-aggregate-quarry-area-raster-eck4-{res}.tif'
    with rio.open(
        out_path, 'w',
        driver='GTiff',
        height=template.shape[0], width=template.shape[1],
        count=1, dtype=grid.dtype,
        crs=template.crs, transform=template.transform,
        compress='lzw',
    ) as dst:
        dst.write(grid, 1)
    print(f"Saved {out_path.name}")

Saved global-aggregate-quarry-area-raster-eck4-1km.tif
